# BIO_0203 · S116 Real Data Diff Expr

**Bioinformatics · transpror · L1 · public seed 1000**

This notebook is locked to the task selected on the RBioBench review website. Run **Setup**, inspect the task-aware data check, write your own R solution, and then run **Check output**. A reference solution is available at the end if you need it.

> Load 'protein_expression.tsv' from the 'inputs' directory. Filter the expression matrix to keep only rows (proteins) that have no missing values (NA). Save this filtered expression matrix to 'outputs/filtered_expression.csv' as a CSV file. Return the filtered expression matrix.

**Expected artifacts:** `outputs/filtered_expression.csv`

## Input data · seed 1000

These are the actual files loaded by **Setup**. Large tables are previewed below.

### `inputs/protein_expression.tsv`

```text
	S01	S02	S03	S04	S05	S06	S07	S08	S09	S10
PROT000	9.559	9.2	4.595	8.803	8.476	11.745	10.31		12.192	6.926
PROT001	10.756	13.747	10.552	15.382	8.085	10.916		10.747		9.113
PROT002	13.366	9.542	11.694	6.892	7.102	7.775	10.598	10.309	9.969	11.481
PROT003	10.755	9.069	10.986	8.208	9.96	10.22	7.729	10.131	5.981	12.659
PROT004	7.868	7.522	8.995	12.133	9.292	14.714		9.576	7.703	
PROT005	13.444	10.462	11.766	11.849	12.865	11.135	12.02	7.879	8.197	10.342
PROT006	7.564	10.81	6.279	14.09		10.08	6.702	7.407		13.182
PROT007	9.985	12.373	7.349	9.859	10.226	9.945	8.905	13.129	14.334	8.781
PROT008	6.827	10.084	10.953	12.147	9.162	9.279	12.79	10.988	8.049	10.559
PROT009	13.697		11.338	6.293	7.869	10.287	8.818	7.103	10.317	7.921
PROT010	9.635	9.221	10.433	11.885	7.389	11.648	11.767	8.723	10.016	6.842
… [first 12 of 51 lines shown]
```

In [ ]:
# @title Setup
task_id <- "BIO_0203"
session_root <- "/content/rbiobench"
archive <- file.path(session_root, "rbiobench-site.tar.gz")
pack_dir <- file.path(session_root, "practice_pack")
workspace <- file.path(session_root, "workspace", task_id)
dir.create(session_root, recursive = TRUE, showWarnings = FALSE)

# The practice pack lives in the GitHub repo under colab/practice_pack.
if (!file.exists(archive)) {
  download.file(
    "https://codeload.github.com/xihaopark/RBioBench-site/tar.gz/refs/heads/main",
    archive, mode = "wb", quiet = TRUE
  )
}

unlink(pack_dir, recursive = TRUE, force = TRUE)
extract_dir <- file.path(session_root, "repo")
unlink(extract_dir, recursive = TRUE, force = TRUE)
untar(archive, exdir = extract_dir)
stopifnot(file.rename(file.path(extract_dir, "RBioBench-site-main", "colab", "practice_pack"), pack_dir))
unlink(extract_dir, recursive = TRUE, force = TRUE)
unlink(workspace, recursive = TRUE, force = TRUE)
dir.create(workspace, recursive = TRUE, showWarnings = FALSE)
status <- system2("python3", c(
  shQuote(file.path(pack_dir, "scripts", "materialize_case.py")),
  "--release", shQuote(pack_dir), "--task", task_id, "--workspace", shQuote(workspace)
))
stopifnot(status == 0)
setwd(workspace)
cat("Ready:", task_id, "\nInputs:", paste(list.files("inputs"), collapse = ", "), "\n")

## Explore the inputs

A short, task-relevant check of the data before you start.

In [ ]:
protein <- read.delim("inputs/protein_expression.tsv", row.names = 1, check.names = FALSE)
missing_per_protein <- rowSums(is.na(protein))
cat(nrow(protein), "proteins ×", ncol(protein), "samples",
    "· overall missing:", round(100 * mean(is.na(protein)), 1), "%",
    "· complete proteins:", sum(missing_per_protein == 0), "\n")
barplot(table(missing_per_protein),
        xlab = "Missing samples per protein", ylab = "Number of proteins",
        main = "Protein-level missingness")

## Your solution

Edit this native-R cell and run it. It should read from `inputs/` and write the required artifact to `outputs/`.

In [ ]:
# Write your R solution below, then run this cell.
dir.create("outputs", showWarnings = FALSE)

# Inputs: inputs/protein_expression.tsv
# Required output: outputs/outputs/filtered_expression.csv
# Your code starts here:


In [ ]:
# @title Check output
score <- system2("python3", c(
  shQuote(file.path(pack_dir, "evaluator", "score_submission.py")),
  "--release", shQuote(pack_dir), "--task", task_id,
  "--outputs", shQuote(file.path(workspace, "outputs"))
), stdout = TRUE, stderr = TRUE)
raw_report <- paste(score, collapse = "\n")
report <- tryCatch(jsonlite::fromJSON(raw_report, simplifyVector = FALSE), error = function(e) NULL)
escape_html <- function(x) {
  x <- gsub("&", "&amp;", as.character(x), fixed = TRUE)
  x <- gsub("<", "&lt;", x, fixed = TRUE)
  x <- gsub(">", "&gt;", x, fixed = TRUE)
  x
}

if (is.null(report)) {
  IRdisplay::display_html(paste0(
    '<div style="border:1px solid #b42318;background:#fff4f2;padding:18px;border-radius:8px">',
    '<div style="font-size:20px;font-weight:700;color:#b42318">CHECK ERROR</div>',
    '<pre style="white-space:pre-wrap">', escape_html(raw_report), '</pre></div>'
  ))
} else if (isTRUE(report$pass)) {
  IRdisplay::display_html(paste0(
    '<div style="border:1px solid #1f7a46;background:#eefbf3;padding:18px;border-radius:8px">',
    '<div style="font-size:22px;font-weight:700;color:#176b3a">✓ PASS</div>',
    '<div style="margin-top:6px;color:#245b3c">All required artifacts match the reference checks.</div></div>'
  ))
} else {
  failures <- unlist(lapply(names(report$comparisons), function(file) {
    item <- report$comparisons[[file]]
    if (isTRUE(item$match)) return(NULL)
    reason <- if (is.null(item$reason)) "Output did not match the reference." else item$reason
    tier <- if (is.null(item$tier)) "" else paste0(' <span style="color:#777">[', escape_html(item$tier), ']</span>')
    paste0('<li style="margin:8px 0"><strong>', escape_html(file), '</strong>: ',
           escape_html(reason), tier, '</li>')
  }))
  IRdisplay::display_html(paste0(
    '<div style="border:1px solid #b42318;background:#fff4f2;padding:18px;border-radius:8px">',
    '<div style="font-size:22px;font-weight:700;color:#b42318">✕ FAIL</div>',
    '<div style="margin-top:6px;color:#7a271a">The output needs attention:</div>',
    '<ul style="margin:8px 0 0;padding-left:22px">', paste(failures, collapse = ""), '</ul></div>'
  ))
}

## Reference solution.R

If you are stuck or want to compare approaches, the exact frozen reference solution is below. Run **Setup** again before using it if you want a clean workspace.

In [ ]:
run_task <- function(work_dir) {
  expr_file <- file.path(work_dir, "inputs", "protein_expression.tsv")
  expr <- read.table(expr_file, sep="\t", header=TRUE, row.names=1)
  
  # Filter no missing
  expr_clean <- expr[rowSums(is.na(expr)) == 0, ]
  
  # Save output
  write.csv(expr_clean, file.path(work_dir, "outputs", "filtered_expression.csv"))
  
  return(expr_clean)
}

run_task(getwd())